In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [8]:
loader = PyPDFLoader("../data/data_science_syllabus.pdf")
docs = loader.load()

In [9]:
len(docs)

10

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)

In [11]:
len(splitted_docs)

10

In [13]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = InMemoryVectorStore.from_documents(splitted_docs, embeddings)

In [20]:
@tool
def retrieval_tool(query:str):
    """
    This tool can help you to retrieve the relevant data of the documents, which has details about the data science syllabus.
    """
    docs = vector_store.similarity_search(query, k=4)
    context = ""
    
    for doc in docs:
        context += doc.page_content + "\n\n"
        
    return context

In [22]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [ ]:
System_prompt = """
    You are a helpful assistant that answers questions using retrieved context. Always use `retrieval_tool` tool for requiring external knowledge.
"""

In [26]:
agent = create_agent(
    model=llm,
    tools=[retrieval_tool],
    system_prompt=System_prompt,
)

In [28]:
query = "What are the Modeule 1 topics?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

In [31]:
result = response["messages"][-1].content

In [36]:
print(result)

[{'type': 'text', 'text': 'The topics covered in **Module 1: Python Programming for Data & AI (4 weeks)** are:\n\n* **Python basics:** variables, loops, conditions, functions\n* **Data structures:** list, dict, set, tuple\n* **File handling, JSON, APIs**\n* **OOPs basics**\n* **Exception handling**\n\n### Additional Details for Module 1:\n* **Tools Used:** Python (Jupyter/Colab), ChatGPT (for code explanation/debugging)\n* **Mini Projects:** \n  * Basic data scraper & analyzer\n  * Python Quiz App\n* **Mock Interview focus:** Python + Logic + File Handling', 'extras': {'signature': 'Ep4GCpsGAWkUfROi28UBnqEMuKdikeY+WyI7217wqc0NuUehhJyZVs8X1RzNMhfoxBjKoC39N4nev220ySsAAo87RU4lygYoIhwWVAqgTQi/Jksgu+bv4CY5TUo1cgp9ruyVzKemjWywlHuhKGsH+M70fRNddzdz5qq+/E1q21oAU8uq/sytSZPGoNZEe5zorrhUZWeQpjl/1N2TrZPg81K+NqUBsi0+dpY2UHEfWRo5FUh1xTyeSC6uiffObVfG7iqPnFUr1niJsj+3ocEOJhfP//diFrHw+jJEQBofOJVR3XLFfGurzqUOPnLsfhLI7sxH+Mr0J9e7K83e4uJg9X08vLVPGGmQ8dCiGUeIuoI8JZCgTbHcplWVBosNWa74Q2DO1mSfSKK9uI5iYShFDy/35i